# 06. Model comparison

All models side by side on the same validation rows and metrics, the final model choice, and the single score on the test set.

## Weighted validation sample: check before use

The LoRA LLM, and possibly TabPFN, are too slow to score all 965,524 validation rows, so they may be scored on all 1,036 laundering rows plus a random 200,000 normal rows, each weighted to stand for 4.8 rows. Before relying on that, the weighted estimate is compared with full validation for the three models already scored in full, over 5 random samples. The rule, fixed before seeing the results: if any weighted PR-AUC differs from the full value by more than 0.01, the slow models are scored on all of validation instead.

In [2]:
# Reload what this section needs from disk, so it runs without the training cells above.
from pathlib import Path

import mlflow
import mlflow.lightgbm
import numpy as np
import pandas as pd

from bankguard.data import load_accounts, load_transactions, split_by_date
from bankguard.evaluate import evaluate, weighted_sample
from bankguard.features import FEATURES, build_features, fit_currency_median
from bankguard.model import predict

DATA = Path("../data")
tx = load_transactions(DATA / "HI-Small_Trans.csv")
accounts = load_accounts(DATA / "HI-Small_accounts.csv")
splits = split_by_date(tx)
train, val = splits["train"], splits["validation"]
X = build_features(tx, accounts, fit_currency_median(train))
X_val, y_val = X.loc[val.index, FEATURES], val["is_laundering"].to_numpy()
del tx, X

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
runs = mlflow.search_runs(
    experiment_names=["bankguard-laundering"],
    filter_string="tags.mlflow.runName = 'LightGBM, owner attributes'",
)
run_id = runs.loc[0, "run_id"]
lgbm = mlflow.lightgbm.load_model(f"runs:/{run_id}/model")
pre = mlflow.artifacts.load_dict(f"runs:/{run_id}/preprocessing.json")

scores = {
    "LightGBM": predict(lgbm, X_val, pre["features"], pre["categories"]),
    "DistilBERT": np.load("../models/distilbert/val_scores.npy"),
    "ModernBERT": np.load("../models/modernbert/modernbert_val_scores.npy"),
}
np.save("../models/lightgbm_val_scores.npy", scores["LightGBM"])
print({name: round(evaluate(y_val, s, name)["pr_auc"], 4) for name, s in scores.items()})

c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'LightGBM': 0.3412, 'DistilBERT': 0.2281, 'ModernBERT': 0.3023}


In [2]:
full = {name: evaluate(y_val, s, name)["pr_auc"] for name, s in scores.items()}

rows = []
for seed in range(5):
    idx, w = weighted_sample(y_val, n_negatives=200_000, seed=seed)
    for name, s in scores.items():
        estimate = evaluate(y_val[idx], s[idx], name, sample_weight=w)["pr_auc"]
        rows.append({"seed": seed, "model": name, "difference": estimate - full[name]})

check = pd.DataFrame(rows)
print(f"largest difference: {check['difference'].abs().max():.4f} (rule: at most 0.0100)")
check.pivot(index="seed", columns="model", values="difference").round(4)

largest difference: 0.0107 (rule: at most 0.0100)


model,DistilBERT,LightGBM,ModernBERT
seed,,,
0,-0.0009,0.0103,0.0004
1,-0.0075,0.0087,-0.0067
2,0.0037,-0.0014,0.0075
3,-0.0026,0.0050,-0.0041
4,-0.0040,0.0065,0.0107


Over 5 random samples, the weighted PR-AUC differed from the full value by at most 0.0107 (ModernBERT, seed 4; LightGBM reached 0.0103 on seed 0), just over the 0.01 limit fixed in advance. Following the rule, models too slow for a single run are scored on all of validation instead, split across separate runs if needed. The differences are small compared with the gaps between models (0.04 to 0.11 PR-AUC), so the weighted sample would not have changed their ranking, but the rule was set before the check and is kept.

## All models on validation

Every model scored on the same 965,524 validation rows with the same metrics, from the two one-line rules up to the sequence model. Feature ablations of LightGBM are in `02_baselines.ipynb`. The sequence model enters as the average of its three seeds, as decided before its results, with the mean of the single seeds shown beside it.

In [3]:
import json

scores["ModernBERT, 2 epochs"] = scores.pop("ModernBERT")
scores["ModernBERT, 4 epochs"] = np.load("../models/modernbert/modernbert_4ep_val_scores.npy")
scores["TabPFN-3.5-Fast"] = np.load("../models/tabpfn/tabpfn_val_scores.npy")
scores["Qwen3-0.6B LoRA"] = np.load("../models/qwen/qwen_val_scores.npy")
scores["Sequence, average of 3 seeds"] = np.load("../models/sequence/sequence_val_scores_avg.npy")

rules = {
    "Rule: every ACH payment": (val["payment_format"] == "ACH").to_numpy().astype(float),
    "Rule: every new counterparty": (
        (X_val["pair_prior_3d"] == 0) & (X_val["self_transfer"] == 0)
    ).to_numpy().astype(float),
}

table = pd.DataFrame([evaluate(y_val, s, name) for name, s in {**rules, **scores}.items()])
seeds = pd.DataFrame(json.loads(Path("../models/sequence/sequence_metrics.json").read_text()))
metric_cols = ["pr_auc", "recall_at_0.1%_fpr", "recall_at_1.0%_fpr"]
single = {"model": "Sequence, single seed (mean of 3)", **seeds[metric_cols].mean()}
table = pd.concat([table, pd.DataFrame([single])], ignore_index=True)
table.sort_values("pr_auc").reset_index(drop=True).round(4)

,model,pr_auc,recall_at_0.1%_fpr,recall_at_1.0%_fpr
0,Rule: every ACH payment,0.0067,0.0000,0.0000
1,Rule: every new counterparty,0.0156,0.0000,0.0000
2,DistilBERT,0.2281,0.2963,0.7259
3,"ModernBERT, 2 epochs",0.3023,0.3600,0.7828
4,Qwen3-0.6B LoRA,0.3060,0.3571,0.7635
5,LightGBM,0.3412,0.3793,0.7635
6,"ModernBERT, 4 epochs",0.3526,0.3755,0.7606
7,TabPFN-3.5-Fast,0.3985,0.3967,0.7963
8,"Sequence, single seed (mean of 3)",0.4568,0.5183,0.8259
9,"Sequence, average of 3 seeds",0.4855,0.5270,0.8388


The two one-line rules reproduce `02_baselines.ipynb` (PR-AUC 0.0067 and 0.0156). They cannot rank transactions, only flag or not, and each flags far more than 1% of normal transactions, so neither fits within a 0.1% or 1% alert budget.

The sequence model leads on every metric: the average of its three seeds reaches PR-AUC 0.486, recall 0.527 at a 0.1% false positive rate and 0.839 at 1%, and a single seed averages 0.457, still well ahead of the next model. TabPFN-3.5-Fast follows at 0.398. ModernBERT after four epochs (0.353) and LightGBM (0.341) are level within LightGBM's spread across seeds. Qwen3-0.6B with LoRA (0.306) is about level with ModernBERT after two epochs (0.302) despite four times the parameters, and DistilBERT is lowest of the trained models (0.228). The models that read the features or the transaction history directly beat every model that reads them as text.

Validation was also used for early stopping, for choosing between the two ModernBERT runs and for this comparison, so these scores are slightly optimistic; the single test set score at the end gives the unbiased figure for the chosen model.

In [4]:
from bankguard.data import load_patterns, typology_for
from bankguard.evaluate import laundering_group, recall_by_group

patterns = load_patterns(DATA / "HI-Small_Patterns.txt")
val_group = laundering_group(typology_for(val, patterns), X_val["sender_out_3d"])

group_table = pd.concat(
    {name: recall_by_group(y_val, s, val_group)["recall_at_1.0%_fpr"] for name, s in scores.items()},
    axis=1,
)
group_table.insert(0, "laundering", recall_by_group(y_val, scores["LightGBM"], val_group)["laundering"])
group_table.round(3)

,laundering,LightGBM,DistilBERT,"ModernBERT, 2 epochs","ModernBERT, 4 epochs",TabPFN-3.5-Fast,Qwen3-0.6B LoRA,"Sequence, average of 3 seeds"
patterned,657,0.913,0.854,0.930,0.896,0.945,0.901,0.989
"unpatterned, quiet sender",246,0.764,0.772,0.813,0.793,0.829,0.809,0.890
"unpatterned, hub sender",133,0.023,0.008,0.000,0.030,0.000,0.000,0.000


At a 1% false positive rate, the sequence model leads on both groups that can be detected: patterned laundering (98.9%, against 94.5% for TabPFN-3.5-Fast and 91.3% for LightGBM) and unpatterned laundering from quiet senders (89.0%, against 82.9% and 76.4%). Qwen3-0.6B with LoRA catches 80.9% of quiet sender laundering, above LightGBM, but stays behind TabPFN and the sequence model on both groups.

Laundering from hub senders is missed by every model, between 0% and 3% recall, across gradient boosting, three fine-tuned text transformers, a tabular foundation model and a sequence model that reads each account's recent history. That seven different learners fail on the same group is strong evidence that neither the 22 features nor the last 64 transactions of an account carry signal for it. With hub sent laundering undetected, recall cannot exceed (657 + 246) / 1,036 = 87.2%; the sequence model's 83.9% is close to that limit, so further gains need new information about hub senders, such as graph structure across many accounts, rather than a different model.

## Combination test and model choice

As fixed in advance, the three models with the highest validation PR-AUC are combined by an equal-weight average of their ranks, with no other combination tried. The model with the higher validation PR-AUC, the combination or the best single model, is chosen and scored once on the test set.

In [5]:
from scipy.stats import rankdata

pr_auc = {name: evaluate(y_val, s, name)["pr_auc"] for name, s in scores.items()}
top3 = sorted(pr_auc, key=pr_auc.get, reverse=True)[:3]
print("top 3:", {name: round(pr_auc[name], 4) for name in top3})

# Each model's scores become ranks from 0 to 1, so models with different score scales count equally.
combo = np.mean([rankdata(scores[name]) / len(y_val) for name in top3], axis=0)
best = top3[0]

choice = pd.DataFrame([
    evaluate(y_val, combo, "Rank average of the top 3"),
    evaluate(y_val, scores[best], best),
])
display(choice.round(4))

display(pd.concat({
    "Rank average of the top 3": recall_by_group(y_val, combo, val_group)["recall_at_1.0%_fpr"],
    best: recall_by_group(y_val, scores[best], val_group)["recall_at_1.0%_fpr"],
}, axis=1).round(3))

chosen = choice.loc[choice["pr_auc"].idxmax(), "model"]
print("chosen model:", chosen)

with mlflow.start_run(run_name="Rank average of the top 3"):
    mlflow.set_tag("components", ", ".join(top3))
    mlflow.log_metrics({k.replace("%", "pct"): v for k, v in choice.iloc[0].items() if k != "model"})

top 3: {'Sequence, average of 3 seeds': 0.4855, 'TabPFN-3.5-Fast': 0.3985, 'ModernBERT, 4 epochs': 0.3526}


,model,pr_auc,recall_at_0.1%_fpr,recall_at_1.0%_fpr
0,Rank average of the top 3,0.4518,0.4653,0.8050
1,"Sequence, average of 3 seeds",0.4855,0.5270,0.8388


,Rank average of the top 3,"Sequence, average of 3 seeds"
patterned,0.953,0.989
"unpatterned, quiet sender",0.841,0.890
"unpatterned, hub sender",0.008,0.000


chosen model: Sequence, average of 3 seeds


The three models with the highest validation PR-AUC are the sequence model (0.486), TabPFN-3.5-Fast (0.398) and ModernBERT after four epochs (0.353). Their equal-weight rank average reaches PR-AUC 0.452, recall 0.465 at a 0.1% false positive rate and 0.805 at 1%, below the sequence model alone on every metric and on both detectable groups (95.3% against 98.9% on patterned laundering, 84.1% against 89.0% on quiet senders). The other two models mostly catch a subset of what the sequence model already catches, and rank it less sharply, so an equal vote dilutes the stronger ranking. A weighted combination could do better but would mean tuning on validation, which the rule excludes. By the rule fixed in advance, the sequence model, as the average of its three seeds, is chosen and scored once on the test set.

## Test set

The model choice above is locked. Each model's test predictions are produced and saved without reading the test labels, then all are evaluated together in one cell below. The sequence model's score is the headline result; the others are reported for reference only and do not change the choice.

In [6]:
tx = load_transactions(DATA / "HI-Small_Trans.csv")
test = split_by_date(tx)["test"]
X = build_features(tx, accounts, fit_currency_median(train))
X_test = X.loc[test.index, FEATURES]
del tx, X

lgbm_test = predict(lgbm, X_test, pre["features"], pre["categories"])
np.save("../models/lightgbm_test_scores.npy", lgbm_test)
print(f"test rows: {len(X_test):,}; scores: {len(lgbm_test):,}; missing: {int(np.isnan(lgbm_test).sum())}")

test rows: 862,792; scores: 862,792; missing: 0


In [2]:
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd

from bankguard.data import (
    load_accounts,
    load_patterns,
    load_transactions,
    split_by_date,
    typology_for,
)
from bankguard.evaluate import evaluate, laundering_group, recall_by_group
from bankguard.features import build_features, fit_currency_median

DATA = Path("../data")
tx = load_transactions(DATA / "HI-Small_Trans.csv")
accounts = load_accounts(DATA / "HI-Small_accounts.csv")
splits = split_by_date(tx)
val, test = splits["validation"], splits["test"]
X = build_features(tx, accounts, fit_currency_median(splits["train"]))
sender_out_3d = X.loc[test.index, "sender_out_3d"]
del tx, X

files = {
    "Sequence, average of 3 seeds (chosen)": "sequence/sequence_{}_scores_avg.npy",
    "LightGBM": "lightgbm_{}_scores.npy",
    "DistilBERT": "distilbert/{}_scores.npy",
}
y_val, y_test = val["is_laundering"].to_numpy(), test["is_laundering"].to_numpy()
test_scores = {name: np.load(f"../models/{path.format('test')}") for name, path in files.items()}
val_pr_auc = {name: evaluate(y_val, np.load(f"../models/{path.format('val')}"), name)["pr_auc"]
              for name, path in files.items()}
assert all(len(s) == len(y_test) for s in test_scores.values())
print(f"test: {len(y_test):,} rows, {int(y_test.sum())} laundering")

test_table = pd.DataFrame([evaluate(y_test, s, name) for name, s in test_scores.items()])
test_table.insert(1, "validation_pr_auc", test_table["model"].map(val_pr_auc))
display(test_table.round(4))

test_group = laundering_group(typology_for(test, load_patterns(DATA / "HI-Small_Patterns.txt")), sender_out_3d)
by_group = pd.concat(
    {name: recall_by_group(y_test, s, test_group)["recall_at_1.0%_fpr"] for name, s in test_scores.items()},
    axis=1,
)
by_group.insert(0, "laundering", recall_by_group(y_test, test_scores["LightGBM"], test_group)["laundering"])
display(by_group.round(3))

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("bankguard-laundering")
with mlflow.start_run(run_name="Test set results"):
    mlflow.set_tag("chosen_model", "Sequence, average of 3 seeds")
    for key, row in zip(["sequence", "lightgbm", "distilbert"], test_table.to_dict("records")):
        mlflow.log_metrics({f"{key}_test_{k.replace('%', 'pct')}": v
                            for k, v in row.items() if k not in ("model", "validation_pr_auc")})

c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


test: 862,792 rows, 956 laundering


,model,validation_pr_auc,pr_auc,recall_at_0.1%_fpr,recall_at_1.0%_fpr
0,"Sequence, average of 3 seeds (chosen)",0.4855,0.3393,0.4561,0.7856
1,LightGBM,0.3412,0.1309,0.1862,0.5157
2,DistilBERT,0.2281,0.0560,0.0931,0.3462


,laundering,"Sequence, average of 3 seeds (chosen)",LightGBM,DistilBERT
patterned,585,0.964,0.670,0.451
"unpatterned, quiet sender",261,0.716,0.387,0.257
"unpatterned, hub sender",110,0.000,0.000,0.000


The test set (9 to 10 Sep) holds 862,792 rows, 956 of them laundering. The chosen model, the sequence model as the average of its three seeds, reaches test PR-AUC 0.339, recall 0.456 at a 0.1% false positive rate and 0.786 at 1%. It catches 96.4% of patterned laundering and 71.6% of unpatterned laundering from quiet senders at a 1% false positive rate, and none of the 110 laundering transactions from hub senders. These are the unbiased headline results of the modelling track.

The ranking from validation holds and the sequence model's lead widens: its test PR-AUC is 2.6 times LightGBM's (0.131), against 1.4 times on validation. All models score much lower on test than on validation, however: the sequence model by 30%, LightGBM by 62% and DistilBERT by 75%. A drop this large and this uneven points to a difference between the validation and test days rather than optimism from model selection alone; the test days lie at the edge of the data, just before the thin 11 to 18 Sep tail. This is examined day by day below.

In [3]:
val_scores = {name: np.load(f"../models/{path.format('val')}") for name, path in files.items()}

rows = []
for split, frame, labels, scores_ in [("validation", val, y_val, val_scores), ("test", test, y_test, test_scores)]:
    days = frame["date"].dt.strftime("%d %b").to_numpy()
    for day in sorted(set(days)):
        m = days == day
        row = {
            "split": split, "day": day, "rows": int(m.sum()),
            "last_time": frame.loc[m, "timestamp"].max().strftime("%H:%M"),
            "laundering": int(labels[m].sum()), "laundering_pct": 100 * labels[m].mean(),
        }
        for name, s in scores_.items():
            row[f"{name.split(',')[0]} PR-AUC"] = evaluate(labels[m], s[m], name)["pr_auc"]
        rows.append(row)

pd.DataFrame(rows).round(4)

,split,day,rows,last_time,laundering,laundering_pct,Sequence PR-AUC,LightGBM PR-AUC,DistilBERT PR-AUC
0,validation,07 Sep,482751,23:59,497,0.1030,0.4938,0.3536,0.2472
1,validation,08 Sep,482773,23:59,539,0.1116,0.4782,0.3297,0.2106
2,test,09 Sep,654467,23:59,514,0.0785,0.2615,0.0712,0.0367
3,test,10 Sep,208325,23:59,442,0.2122,0.4766,0.3891,0.2029


By day, the drop is confined to 9 Sep. On 10 Sep every model scores about as on validation (sequence model 0.477, LightGBM 0.389, DistilBERT 0.203), while on 9 Sep all three collapse (0.262, 0.071 and 0.037). 9 Sep holds 654,467 rows against about 483,000 on every validation day, with a lower laundering share (0.079% against about 0.11%), and 10 Sep holds only 208,325 rows with a higher share (0.21%). A lower laundering share alone would lower PR-AUC by about a quarter, not fivefold as for LightGBM, so the extra 9 Sep rows must be normal transactions that the models score as suspicious.

In [4]:
normal = pd.concat([val, test])
normal = normal[normal["is_laundering"] == 0]
day = normal["date"].dt.strftime("%d %b")

summary = pd.DataFrame({
    "normal rows": day.value_counts().sort_index(),
    "self transfer %": normal.groupby(day)["self_transfer"].mean() * 100,
    "distinct senders": normal.groupby(day)["from_account"].nunique(),
})
display(summary.round(2))
display((pd.crosstab(normal["payment_format"], day, normalize="columns") * 100).round(1))
pd.crosstab(normal["timestamp"].dt.hour, day)

,normal rows,self transfer %,distinct senders
date,,,
07 Sep,482254,2.15,120439
08 Sep,482234,2.16,120464
09 Sep,653953,2.03,205570
10 Sep,207883,2.11,76340


date,07 Sep,08 Sep,09 Sep,10 Sep
payment_format,,,,
ACH,12.0,12.0,13.5,14.4
Bitcoin,3.0,3.0,2.8,2.9
Cash,10.7,10.7,10.9,9.7
Cheque,41.2,41.2,40.9,38.2
Credit Card,29.3,29.3,28.3,31.2
Wire,3.8,3.8,3.5,3.5


date,07 Sep,08 Sep,09 Sep,10 Sep
timestamp,,,,
0,34038,33951,41268,22578
1,19890,19346,26848,8265
2,19142,19350,26600,8175
3,19409,19589,26334,8058
4,19315,19497,26568,8425
5,19712,19379,26871,8124
6,19960,19181,26856,8088
7,19505,19372,26500,8192
8,19435,19705,26970,8233


The extra rows on 9 Sep are spread evenly across every hour (about 26,600 normal rows per hour against about 19,500 on the validation days), so they are not a burst. They come from far more accounts: 205,570 distinct senders against about 120,450 on a normal day. The payment format mix and the share of self transfers barely change. 9 Sep therefore looks like a wave of occasionally active accounts, whose short histories and new counterparties resemble laundering behaviour and fill the top of the alert list with false positives; 10 Sep, with fewer rows and senders and a higher laundering share, matches the thinning of the data before the 11 to 18 Sep tail. This is a shift in the data at the end of the simulation rather than a failure of the models to generalise to later days, and it is the kind of drift that the planned drift monitoring is meant to detect. The sequence model, which reads each account's recent history instead of trailing counts, degrades least: on 9 Sep it keeps 3.7 times LightGBM's PR-AUC.